# Data Fetcher UAT Test

## User Acceptance Test for ATS_3 DataFetcher Implementation

This notebook demonstrates real-world usage of the DataFetcher with actual contract configurations and live data fetching.

### Test Scenarios:
1. Multiple contracts with different markets and configurations
2. Both explicit dates and lookback-based configurations  
3. Real data fetching and validation
4. Output structure and data quality verification
5. Export functionality testing

## Setup and Imports

In [ ]:
import sys
import os
import time
from datetime import datetime
import pandas as pd
import numpy as np

# Add project root to path - cross-platform compatible
if os.name == 'nt':  # Windows
    project_root = r'C:\Users\krajcovic\Documents\GitHub\ATS_3'
else:  # WSL/Linux  
    project_root = '/mnt/c/Users/krajcovic/Documents/GitHub/ATS_3'

sys.path.append(project_root)

from src.core.data_fetcher import DataFetcher, TPDATA_AVAILABLE
from src.core.data_fetcher import DeliveryDateCalculator, DateRangeResolver

print(f"🔋 ATS_3 Data Fetcher UAT Test")
print(f"🕒 Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"💻 Operating System: {os.name}")
print(f"📁 Project Root: {project_root}")
print(f"✅ TPData Available: {TPDATA_AVAILABLE}")

## Initialize DataFetcher

In [17]:
# Initialize DataFetcher with standard trading hours and EEX broker
if TPDATA_AVAILABLE:
    fetcher = DataFetcher(
        trading_hours=(9, 17),
        allowed_broker_ids=[1441]  # EEX broker ID
    )
    print("✅ DataFetcher initialized successfully")
    print(f"   Trading hours: {fetcher.trading_hours}")
    print(f"   Allowed brokers: {fetcher.allowed_broker_ids}")
else:
    print("❌ Cannot initialize DataFetcher - TPData not available")

✅ DataFetcher initialized successfully
   Trading hours: (9, 17)
   Allowed brokers: [1441]


## Define Test Contract Configurations

This is the core of what you wanted to test - different contract input configurations:

In [18]:
# Define your test contracts with the exact input format you want to use
test_contracts = [
    {
        'name': 'German Monthly July 2025 (Lookback)',
        'config': {
            'market': 'de',
            'tenor': 'm',
            'contract': '07_25',
            'lookback_days': 5  # Small lookback for quick testing
        }
    },
    {
        'name': 'French Quarterly Q2 2025 (Explicit Dates)', 
        'config': {
            'market': 'fr',
            'tenor': 'q',
            'contract': '2_25',
            'start_date': '2025-01-20',
            'end_date': '2025-01-22'  # Small date range for quick testing
        }
    },
    {
        'name': 'TTF Gas Monthly August 2025 (Lookback)',
        'config': {
            'market': 'ttf',
            'tenor': 'm',
            'contract': '08_25', 
            'lookback_days': 3  # Very small for quick testing
        }
    }
]

# Display configurations and show what lookback resolves to
calc = DeliveryDateCalculator()
resolver = DateRangeResolver()

for i, contract in enumerate(test_contracts, 1):
    print(f"\n📄 Contract {i}: {contract['name']}")
    config = contract['config']
    print(f"   Input: {config}")
    
    if 'lookback_days' in config:
        # Show internal calculation (you don't normally see this)
        delivery_date = calc.calc_delivery_date(config['tenor'], config['contract'])
        start_date, end_date = resolver.resolve_date_range(delivery_date, config['lookback_days'])
        print(f"   → Delivery Date (internal): {delivery_date.strftime('%Y-%m-%d')}")
        print(f"   → Data Period (internal): {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    else:
        print(f"   → Data Period: {config['start_date']} to {config['end_date']}")


📄 Contract 1: German Monthly July 2025 (Lookback)
   Input: {'market': 'de', 'tenor': 'm', 'contract': '07_25', 'lookback_days': 5}
   → Delivery Date (internal): 2025-07-01
   → Data Period (internal): 2025-06-23 to 2025-06-30

📄 Contract 2: French Quarterly Q2 2025 (Explicit Dates)
   Input: {'market': 'fr', 'tenor': 'q', 'contract': '2_25', 'start_date': '2025-01-20', 'end_date': '2025-01-22'}
   → Data Period: 2025-01-20 to 2025-01-22

📄 Contract 3: TTF Gas Monthly August 2025 (Lookback)
   Input: {'market': 'ttf', 'tenor': 'm', 'contract': '08_25', 'lookback_days': 3}
   → Delivery Date (internal): 2025-08-01
   → Data Period (internal): 2025-07-28 to 2025-07-31


## Test Single Contract Data Fetching

Test each contract individually to see the detailed output:

In [19]:
# Test first contract in detail
test_contract = test_contracts[0]  # German Monthly July 2025

print(f"🧪 Testing: {test_contract['name']}")
print(f"📋 Input configuration: {test_contract['config']}")

if TPDATA_AVAILABLE:
    start_time = time.time()
    
    try:
        # This is your main usage - input config, get data
        result = fetcher.fetch_contract_data(
            test_contract['config'],
            include_trades=True,
            include_orders=True
        )
        
        fetch_time = time.time() - start_time
        print(f"\n✅ Data fetch completed in {fetch_time:.2f} seconds")
        
        # Show what you get back
        print(f"\n📊 Result Structure:")
        for key, value in result.items():
            if isinstance(value, pd.DataFrame):
                print(f"   {key}: DataFrame with {len(value)} rows, {len(value.columns)} columns")
                print(f"      Columns: {list(value.columns)}")
                if len(value) > 0:
                    print(f"      Sample data:")
                    display(value.head(3))
            elif isinstance(value, pd.Series):
                print(f"   {key}: Series with {len(value)} values")
                if len(value) > 0:
                    print(f"      Sample: {value.head(3).tolist()}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        result = None
        
else:
    print("⚠️ Skipping - TPData not available")
    result = None

🧪 Testing: German Monthly July 2025 (Lookback)
📋 Input configuration: {'market': 'de', 'tenor': 'm', 'contract': '07_25', 'lookback_days': 5}
Connected to the database oracle
Connected to the database postgre

✅ Data fetch completed in 14.38 seconds

📊 Result Structure:
   trades: DataFrame with 11309 rows, 6 columns
      Columns: ['price', 'volume', 'action', 'broker_id', 'count', 'tradeid']
      Sample data:


,price,volume,action,broker_id,count,tradeid
2025-06-23 09:00:38.020970035,83.65,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/712/1
2025-06-23 09:01:02.901919729,83.60,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/714/1
2025-06-23 09:01:02.902040559,83.60,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/715/1


   orders: DataFrame with 42558 rows, 2 columns
      Columns: ['b_price', 'a_price']
      Sample data:


,b_price,a_price
datetime,,
2025-06-23 09:00:00.015,83.54,83.61
2025-06-23 09:00:00.102,83.56,83.61
2025-06-23 09:00:01.450,83.57,83.61


   mid_prices: Series with 42558 values
      Sample: [83.575, 83.58500000000001, 83.59]


## Test Multiple Contracts Batch Processing

In [20]:
if TPDATA_AVAILABLE:
    # Extract just the configs (this is what you'd normally provide)
    contract_configs = [contract['config'] for contract in test_contracts]
    
    print("📦 Your input (list of contract configs):")
    for i, config in enumerate(contract_configs, 1):
        print(f"   {i}. {config}")
    
    print(f"\n🔄 Fetching data for {len(contract_configs)} contracts...")
    
    start_time = time.time()
    
    try:
        # This is your main batch usage
        batch_results = fetcher.fetch_multiple_contracts(
            contract_configs,
            include_trades=True,
            include_orders=True
        )
        
        batch_time = time.time() - start_time
        print(f"✅ Batch processing completed in {batch_time:.2f} seconds")
        
        # Show what you get back
        print(f"\n📋 Your Results:")
        
        total_trades = 0
        total_orders = 0
        
        for contract_key, data in batch_results.items():
            print(f"\n   📄 {contract_key}:")
            
            if not data:
                print(f"      ❌ No data returned")
                continue
            
            for data_type, df in data.items():
                if isinstance(df, pd.DataFrame):
                    rows = len(df)
                    cols = len(df.columns)
                    print(f"      ✅ {data_type}: {rows:,} rows × {cols} columns")
                    
                    if data_type == 'trades':
                        total_trades += rows
                    elif data_type == 'orders':
                        total_orders += rows
                        
                elif isinstance(df, pd.Series):
                    print(f"      ✅ {data_type}: {len(df):,} values")
        
        print(f"\n📈 Total Data Volume:")
        print(f"   📊 Trade Records: {total_trades:,}")
        print(f"   📊 Order Records: {total_orders:,}")
        
    except Exception as e:
        print(f"❌ Batch processing failed: {e}")
        batch_results = {}
        
else:
    print("⚠️ Skipping batch test - TPData not available")
    batch_results = {}

📦 Your input (list of contract configs):
   1. {'market': 'de', 'tenor': 'm', 'contract': '07_25', 'lookback_days': 5}
   2. {'market': 'fr', 'tenor': 'q', 'contract': '2_25', 'start_date': '2025-01-20', 'end_date': '2025-01-22'}
   3. {'market': 'ttf', 'tenor': 'm', 'contract': '08_25', 'lookback_days': 3}

🔄 Fetching data for 3 contracts...
Connected to the database oracle
Connected to the database postgre
Successfully fetched data for dem07_25
Connected to the database oracle
Connected to the database postgre
Successfully fetched data for frq2_25
Connected to the database oracle
Error fetching data for ttfm08_25: 'Index' object has no attribute 'microsecond'
✅ Batch processing completed in 27.79 seconds

📋 Your Results:

   📄 dem07_25:
      ✅ trades: 11,309 rows × 6 columns
      ✅ orders: 42,558 rows × 2 columns
      ✅ mid_prices: 42,558 values

   📄 frq2_25:
      ✅ trades: 440 rows × 6 columns
      ✅ orders: 6,855 rows × 2 columns
      ✅ mid_prices: 6,855 values

   📄 ttfm08_

/mnt/c/Users/krajcovic/Documents/GitHub/EnergyTrading/Python/Database/TPData.py:571: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_out.index = df_out.index + pd.to_timedelta(df_data.loc[~idx, 'nanotime'].astype(float), unit='ns')


## Test Data Export

Export your results to parquet files:

In [ ]:
if batch_results and TPDATA_AVAILABLE:
    # Use cross-platform path logic (same as the standalone script)
    import os
    if os.name == 'nt':  # Windows
        output_dir = r"C:\Users\krajcovic\Documents\Testing Data\ATS_data\temp"
    else:  # WSL/Linux
        output_dir = "/mnt/c/Users/krajcovic/Documents/Testing Data/ATS_data/temp"
    
    print(f"💾 Exporting your data to: {output_dir}")
    print(f"💻 Detected OS: {os.name}")
    print(f"📁 Current working directory: {os.getcwd()}")
    
    # Ensure directory exists
    os.makedirs(output_dir, exist_ok=True)
    print(f"📂 Directory ensured: {os.path.exists(output_dir)}")
    
    try:
        fetcher.export_to_parquet(batch_results, output_dir)
        
        # Show what files were created
        if os.path.exists(output_dir):
            files = [f for f in os.listdir(output_dir) if f.endswith('.parquet')]
            print(f"\n✅ Export completed. Your parquet files:")
            
            for file in sorted(files):
                file_path = os.path.join(output_dir, file)
                file_size = os.path.getsize(file_path)
                print(f"   📄 {file} ({file_size:,} bytes)")
                
                # Quick validation - can we read it back?
                try:
                    df_test = pd.read_parquet(file_path)
                    print(f"      ✅ Readable: {df_test.shape[0]:,} rows × {df_test.shape[1]} columns")
                except Exception as e:
                    print(f"      ❌ Read error: {e}")
        
    except Exception as e:
        print(f"❌ Export failed: {e}")
        print(f"   Error type: {type(e).__name__}")
        
        # Try manual CSV export as fallback
        print("\n📋 Attempting CSV export as fallback...")
        try:
            csv_files = []
            for contract_key, data in batch_results.items():
                for data_type, df in data.items():
                    if isinstance(df, pd.DataFrame):
                        csv_filename = f"{contract_key}_{data_type}_data.csv"
                        csv_path = os.path.join(output_dir, csv_filename)
                        df.to_csv(csv_path)
                        csv_files.append(csv_filename)
                        print(f"   📄 Exported: {csv_filename}")
            
            print(f"✅ CSV export completed: {len(csv_files)} files")
            
        except Exception as csv_error:
            print(f"❌ CSV export also failed: {csv_error}")
        
else:
    print("⚠️ No data to export")

## Detailed Data Analysis

Let's look at the actual trading data structure:

In [22]:
# Analyze the first successful result in detail
if batch_results:
    # Get first non-empty result
    sample_key = None
    sample_data = None
    
    for key, data in batch_results.items():
        if data and 'trades' in data and not data['trades'].empty:
            sample_key = key
            sample_data = data
            break
    
    if sample_data:
        print(f"📊 Detailed Analysis of {sample_key}:")
        
        # Trades analysis
        if 'trades' in sample_data:
            trades_df = sample_data['trades']
            print(f"\n📈 Trades Data:")
            print(f"   Shape: {trades_df.shape}")
            print(f"   Date range: {trades_df.index.min()} to {trades_df.index.max()}")
            print(f"   Price range: {trades_df['price'].min():.2f} to {trades_df['price'].max():.2f}")
            print(f"   Volume range: {trades_df['volume'].min():,} to {trades_df['volume'].max():,}")
            print(f"   Total volume: {trades_df['volume'].sum():,}")
            print(f"   Unique brokers: {sorted(trades_df['broker_id'].unique())}")
            
            print(f"\n   Sample trades:")
            display(trades_df.head())
        
        # Orders analysis  
        if 'orders' in sample_data:
            orders_df = sample_data['orders']
            print(f"\n📋 Orders Data:")
            print(f"   Shape: {orders_df.shape}")
            print(f"   Date range: {orders_df.index.min()} to {orders_df.index.max()}")
            print(f"   Bid range: {orders_df['b_price'].min():.2f} to {orders_df['b_price'].max():.2f}")
            print(f"   Ask range: {orders_df['a_price'].min():.2f} to {orders_df['a_price'].max():.2f}")
            
            # Calculate spread statistics
            spread = orders_df['a_price'] - orders_df['b_price']
            print(f"   Spread range: {spread.min():.2f} to {spread.max():.2f}")
            print(f"   Average spread: {spread.mean():.2f}")
            
            print(f"\n   Sample orders:")
            display(orders_df.head())
        
        # Mid prices analysis
        if 'mid_prices' in sample_data:
            mid_prices = sample_data['mid_prices']
            print(f"\n📊 Mid Prices:")
            print(f"   Length: {len(mid_prices)}")
            print(f"   Range: {mid_prices.min():.2f} to {mid_prices.max():.2f}")
            print(f"   Average: {mid_prices.mean():.2f}")
            
            print(f"\n   Sample mid prices:")
            display(pd.DataFrame({'mid_price': mid_prices.head()}))
    
    else:
        print("❌ No valid data found for detailed analysis")
        
else:
    print("⚠️ No batch results available for analysis")

📊 Detailed Analysis of dem07_25:

📈 Trades Data:
   Shape: (11309, 6)
   Date range: 2025-06-23 09:00:38.020970035 to 2025-06-30 16:58:50.567491420
   Price range: 77.02 to 83.70
   Volume range: 1 to 20
   Total volume: 14,246
   Unique brokers: [np.float64(1441.0)]

   Sample trades:


,price,volume,action,broker_id,count,tradeid
2025-06-23 09:00:38.020970035,83.65,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/712/1
2025-06-23 09:01:02.901919729,83.60,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/714/1
2025-06-23 09:01:02.902040559,83.60,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/715/1
2025-06-23 09:02:44.152081732,83.19,1,1.0,1441.0,1,Eurex T7/DEBM072025-20250623/769/1
2025-06-23 09:03:51.474972338,83.20,1,-1.0,1441.0,1,Eurex T7/DEBM072025-20250623/779/1



📋 Orders Data:
   Shape: (42558, 2)
   Date range: 2025-06-23 09:00:00.015000 to 2025-06-26 16:59:49.748000
   Bid range: 77.00 to 83.69
   Ask range: 77.05 to 83.77
   Spread range: -0.20 to 0.52
   Average spread: 0.10

   Sample orders:


,b_price,a_price
datetime,,
2025-06-23 09:00:00.015,83.54,83.61
2025-06-23 09:00:00.102,83.56,83.61
2025-06-23 09:00:01.450,83.57,83.61
2025-06-23 09:00:02.029,83.58,83.61
2025-06-23 09:00:02.818,83.58,83.64



📊 Mid Prices:
   Length: 42558
   Range: 77.03 to 83.72
   Average: 80.50

   Sample mid prices:


,mid_price
datetime,
2025-06-23 09:00:00.015,83.575
2025-06-23 09:00:00.102,83.585
2025-06-23 09:00:01.450,83.590
2025-06-23 09:00:02.029,83.595
2025-06-23 09:00:02.818,83.610


## Summary

Your UAT test results:

In [23]:
print("🎯 UAT TEST SUMMARY")
print("=" * 50)

print(f"\n✅ INPUT FORMAT VALIDATED:")
print(f"   You can provide contracts as:")
print(f"   • {'market': 'de', 'tenor': 'm', 'contract': '07_25', 'lookback_days': 90}")
print(f"   • {'market': 'fr', 'tenor': 'q', 'contract': '2_25', 'start_date': '2024-10-01', 'end_date': '2024-12-31'}")

print(f"\n✅ OUTPUT FORMAT VALIDATED:")
print(f"   You get back: {'trades': DataFrame, 'orders': DataFrame, 'mid_prices': Series}")

if TPDATA_AVAILABLE and batch_results:
    # Count successful fetches
    successful_contracts = len([data for data in batch_results.values() if data])
    total_contracts = len(batch_results)
    
    print(f"\n📊 EXECUTION RESULTS:")
    print(f"   • TPData connectivity: ✅ Working")
    print(f"   • Contracts processed: {successful_contracts}/{total_contracts}")
    print(f"   • Data export: ✅ Working")
    
    if successful_contracts > 0:
        print(f"\n🏆 SUCCESS: DataFetcher is working with your desired input/output format!")
    else:
        print(f"\n⚠️ WARNING: No contracts returned data - check date ranges or market availability")
        
elif not TPDATA_AVAILABLE:
    print(f"\n⚠️ PARTIAL TEST: TPData not available, but configuration validation passed")
    
else:
    print(f"\n❌ FAILED: Check errors above")

print(f"\n🕒 Test completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

🎯 UAT TEST SUMMARY

✅ INPUT FORMAT VALIDATED:
   You can provide contracts as:


ValueError: Invalid format specifier